# Nemotron Super V3 GRPO/DAPO Training with NemoRL

## Overview
This guide demonstrates the step-by-step RL training of the Nemotron Super V3 model on an NVIDIA B200 GPU cluster or a [GB200-NVL72](https://www.nvidia.com/en-au/data-center/gb200-nvl72/) rack system running Slurm.

We will carry out GRPO/DAPO training of the model on the [DAPO-Math-17k dataset](https://huggingface.co/datasets/BytedTsinghua-SIA/DAPO-Math-17k). This is a single-domain reinforcement learning example with verifiable rewards.

**Notes**:
- Due to the distributed nature of the setup and training steps, all commands in this notebook are intended to be copied, pasted, and executed in the relevant interactive Docker shell environment on either the head node or worker nodes, rather than from within a single JupyterLab environment.

- Interactive vs. batch training: in a production setting, it is more convenient to submit training jobs as Slurm batch jobs. However, setting up an interactive training environment allows you to iterate and debug faster. Once the interactive jobs run smoothly, you can submit them as batch training jobs.

- We start this tutorial with a batch submission guide, followed by a step-by-step guide to set up an interactive training cluster.


## Prerequisites

- **Compute**: 3xB200 nodes (each with 8xGPUs, i.e. 24xB200 GPUs in total) with infiniband connection, or 5xGB200 nodes (each with 4xGPUs, i.e. 20x GB200 GPUs in total) on a single GB200 NVL72 rack. Note that B200/GB200 GPUs have ~183/189 GB of HBM, which is not sufficient for full-weight co-located RL training (i.e., both training the policy model and running rollout on the same set of GPUs), so we use non-colocated training, with 1 node dedicated to rollout and 4 nodes for policy training.

    On a Slurm system, you can check the availability of GB200 nodes with something similar to:

    ```
    sinfo|grep gb200nvl72
    ```

    Replace "gb200nvl72" with your correct GB200 Slurm partition name. Then request a specific interactive node with:

    ```
    srun -p gb200nvl72 -w gb200-001-compute09 -t 08:00:00 --pty bash
    ```

- **Storage**: A high-speed shared network file system for storing code, models, checkpoints, and other temporary assets. In this guide, we will assume that the shared storage is at `</YOUR/SHARED/NETWORK/STORAGE>` on the host system, to be mounted as `/shared` into the working Docker container, accessible from all nodes. We will also assume the following directory structure

```
</YOUR/SHARED/NETWORK/STORAGE> (on host):/shared (inside container)
|______code
|        |____RL  # NemoRL root directory
|        |____Nemotron/usage-cookbook/Nemotron-3-Super  # Repository containing this notebook
|_______models
|        |____NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16 # base model checkpoint
|_______checkpoints
|_______HF_HOME   # HuggingFace cache directory
```

Note that each model checkpoint (including model's weights and optimizer's state) takes up to ~1Tb of storage. You should also account for the number of checkpoints you would like to keep (e.g., best k=3 checkpoints in the NemoRL training config). In addition, the base BF16 model checkpoint requires ~231Gb of storage, and another ~231Gb for the Megatron-converted checkpoint.

- **Model**: download the HuggingFace-format model with the [HF CLI tool](https://huggingface.co/docs/huggingface_hub/en/guides/cli) to the shared location on the high-speed storage. In this example, we will start from a Nemotron Super-V3 pretrained model [https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16) fresh out of pretraining. The DAPO training process can help the model discover advanced math reasoning entirely by itself, aka. the Deepseek "aha" [moment](https://www.reddit.com/r/OpenAI/comments/1i6jsr2/deepseek_discovered_their_new_model_having_an_aha). 

In [ ]:
mkdir -p </YOUR/SHARED/NETWORK/STORAGE>/models
cd </YOUR/SHARED/NETWORK/STORAGE>/models
export HF_TOKEN=<YOUR_HF_TOKEN>

hf download nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16 --local-dir NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16

- **Docker image**: Build the NemoRL Docker container

Reserve a B200/GB200 compute node. From here, clone and check out the [NemoRL Super-v3 branch](https://github.com/NVIDIA-NeMo/RL/blob/super-v3/docs/guides/nemotron-3-super.md).

In [ ]:
mkdir -p </YOUR/SHARED/NETWORK/STORAGE>/code  # Put the correct path to your shared storage here
cd </YOUR/SHARED/NETWORK/STORAGE>/code

git clone --recursive -b super-v3 https://github.com/NVIDIA-NeMo/RL.git  # checkout the Nemotron Super-V3 branch, then checkout the submodules after

Next, from the NemoRL repo root directory, build the Docker container and push it to a central registry accessible to all nodes, such as NVIDIA NGC or Docker Hub. 

In [ ]:
cd </YOUR/SHARED/NETWORK/STORAGE>/code/RL
docker buildx build \
  --build-arg SKIP_SGLANG_BUILD=1 \
  --build-arg BUILD_CUSTOM_VLLM=1 \
  --target release \
  --build-context nemo-rl=. \
  -f docker/Dockerfile \
  --tag nvcr.io/<YOUR_NGC_ORG>/nemo-rl-gb200:superv3-a90de923d \
  --push .

**Note**: In this example, we make use of the NVIDIA Cloud Registry NGC. See the NGC user's [guide](https://docs.nvidia.com/ngc/latest/ngc-user-guide.html) on how to set up your account and team, obtain the NGC API key for Docker login, and upload an image to NGC. 
Replace `nvcr.io/<YOUR_NGC_ORG>/nemo-rl-gb200:superv3-a90de923d` with your correct NGC organization, or else use an accessible Docker hub tag.

Also, see the latest NemoRL Docker build [guide](https://github.com/NVIDIA-NeMo/RL/blob/main/docs/docker.md) for more information.

## Step 1. Prepare the training config file

This step defines the training recipe for a single-domain RL workload: verifiable mathematical reasoning on DAPO-style data. The policy learns from sampled solutions, and rewards are computed by math verifiers, so this setup is ideal for tasks where correctness can be programmatically checked (e.g., arithmetic/algebra word problems and competition-style math).


### GB200 Config
In this part, the YAML below is tuned for a 5xGB200 nodes (4xGPUs per node, 20 in total), non-colocated setup:
- **1 node** for vLLM generation (rollout)
- **4 nodes** for Megatron policy training

A practical workflow is to first run a short sanity pass (small `max_num_steps`) to validate cluster/config correctness, then scale up sequence length and rollout volume once stable.

**Notes**: 
- The configuration file must be written or copied under the NemoRL root directory, e.g. at `</YOUR/SHARED/NETWORK/STORAGE>/code/RL/examples/configs/recipes/llm/` on your host file system.
- ALL PATHS INSIDE THE CONFIG FILES ARE PATH AS MOUNTED INSIDE THE CONTAINER

In [ ]:
%%writefile </YOUR/SHARED/NETWORK/STORAGE>/code/RL/examples/configs/recipes/llm/dapo17k_nemotron_super_120b_gb200_5x4_noncolocated.yaml

#############################################################
# DAPO-style run on DAPO-Math-17k (pure NeMoRL) - NON-COLOCATED.
#
# - Entry point:
#     uv run examples/run_grpo_math.py --config <this.yaml>
# - Dataset:
#     HuggingFace BytedTsinghua-SIA/DAPO-Math-17k (train)
#     HuggingFace BytedTsinghua-SIA/AIME-2024 (validation)
#   via built-in NeMoRL dataset_name=DAPOMath17K
# - Cluster: 5 nodes × 4 GPUs/node (20 GPUs total)
#   - 1 node (4 GPUs) for vLLM generation
#   - 4 nodes (16 GPUs) for Megatron training
# - Model: override via CLI, e.g.
#     policy.model_name=/shared/models/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16
#     NOTE: ALL PATHS INSIDE THIS CONFIG FILE ARE PATHS AS MOUNTED INSIDE THE CONTAINER
# Notes:
# - Non-colocated setup avoids memory contention between vLLM and Megatron
# - vLLM can use higher GPU memory utilization (0.7-0.8) since not sharing with training
# - DAPO features are GRPO knobs: dynamic sampling + reward shaping (+ optional scaling).
#############################################################

defaults: ../../grpo_math_1B.yaml

grpo:
  max_num_epochs: 1
  max_num_steps: 2000

  # Rollout shape
  num_prompts_per_step: 32
  num_generations_per_prompt: 16
  max_rollout_turns: 1

  # GRPO/DAPO basics
  normalize_rewards: true
  use_leave_one_out_baseline: true
  seed: 42

  # Validation
  val_at_start: true
  val_period: 5
  # Must be non-null for `examples/run_grpo_math.py`.
  # If the validation dataset is shorter than this, it will just run through the dataset once.
  max_val_samples: 256
  val_batch_size: 32

  # ----- DAPO features -----
  use_dynamic_sampling: true
  # Dataloader batch size = batch_multiplier × num_prompts_per_step
  batch_multiplier: 2
  dynamic_sampling_max_gen_batches: 10

  # Overlong reward shaping (penalize excessively long generations)
  reward_shaping:
    enabled: true
    overlong_buffer_length: 512
    overlong_buffer_penalty: 1.0
    max_response_length: ${policy.max_total_sequence_length}

  # Optional linear reward scaling (keeps reward magnitude stable)
  reward_scaling:
    enabled: true
    source_min: 0.0
    source_max: 1.0
    target_min: -1.0
    target_max: 1.0

  # Safety rails
  overlong_filtering: false
  seq_logprob_error_threshold: 2

loss_fn:
  # DAPO clip-higher: set max > min.
  ratio_clip_min: 0.2
  ratio_clip_max: 0.28
  ratio_clip_c: null

  # Generally recommended for long-CoT RL
  token_level_loss: true

  # Keep it simple for getting started
  reference_policy_kl_penalty: 0.0
  reference_policy_kl_type: "k3"
  kl_input_clamp_value: 20.0
  kl_output_clamp_value: 10.0

  use_on_policy_kl_approximation: false
  truncated_importance_sampling_ratio: null
  use_importance_sampling_correction: false
  force_on_policy_ratio: false

checkpointing:
  enabled: true
  checkpoint_dir: "/shared/checkpoints/nemotron-superv3-dapo_math17k"
  metric_name: "val:accuracy"
  higher_is_better: true
  keep_top_k: 3
  save_period: 5
  checkpoint_must_save_by: null

policy:
  model_name: "/shared/models/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16"
  tokenizer:
    name: ${policy.model_name}
  hf_config_overrides: {}

  # Keep batch sizes tied to rollout count (stable for GRPO)
  train_global_batch_size: ${mul:${grpo.num_prompts_per_step}, ${grpo.num_generations_per_prompt}}
  train_micro_batch_size: 1
  logprob_batch_size: 1

  # Long-context RL can be expensive; start smaller and scale up.
  max_total_sequence_length: 1024
  precision: "bfloat16"
  logprob_chunk_size: 512

  # For very large models, Megatron training is typically required.
  megatron_cfg:
    enabled: true
    error_path: "/shared/checkpoints/nemotron-superv3-dapo_math17k/logs/megatron_error.log"
    # Non-colocated: can use less aggressive memory management
    empty_unused_memory_level: 2
    activation_checkpointing: true
    tensor_model_parallel_size: 4
    pipeline_model_parallel_size: 1
    context_parallel_size: 4
    expert_tensor_parallel_size: 4
    expert_model_parallel_size: 4
    pipeline_dtype: ${policy.precision}
    sequence_parallel: true

    apply_rope_fusion: true
    defer_fp32_logits: true
    bias_activation_fusion: false

    optimizer:
      optimizer: "adam"
      lr: 1.0e-6
      min_lr: 1.0e-6
      weight_decay: 0.0
      bf16: true
      fp16: false
      params_dtype: "bfloat16"
      adam_beta1: 0.9
      adam_beta2: 0.999
      adam_eps: 1e-8
      use_distributed_optimizer: true
      use_precision_aware_optimizer: true
      clip_grad: ${policy.max_grad_norm}
      optimizer_cpu_offload: false
      optimizer_offload_fraction: 0.0

    scheduler:
      lr_decay_style: "constant"
      lr_warmup_iters: 20
      lr_warmup_init: 1.0e-7

    distributed_data_parallel_config:
      grad_reduce_in_fp32: false
      # Workaround for Megatron distributed optimizer assertion during logprob fprop:
      # `param_and_grad_buffer.py:261 assert self.param_gather_handle is None`.
      # Disabling overlap avoids starting a new param gather while a prior one is still in-flight.
      overlap_grad_reduce: false
      overlap_param_gather: false
      use_custom_fsdp: false
      data_parallel_sharding_strategy: "optim_grads_params"

  # Disable dtensor path for this getting-started config
  dtensor_cfg:
    _v2: true
    enabled: false

  sequence_packing:
    enabled: true
    train_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.train_micro_batch_size}}
    logprob_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.logprob_batch_size}}
    algorithm: "modified_first_fit_decreasing"
    sequence_length_round: 64

  make_sequence_length_divisible_by: ${policy.megatron_cfg.tensor_model_parallel_size}
  max_grad_norm: 1.0

  generation:
    backend: "vllm"
    max_new_tokens: ${policy.max_total_sequence_length}
    temperature: 1.0
    top_p: 1.0
    top_k: null
    stop_token_ids: null
    stop_strings: null
    vllm_cfg:
      async_engine: true
      precision: ${policy.precision}
      # vLLM on dedicated node: can use TP=4 for better performance
      # With 1 node (4 GPUs), TP=4 uses all GPUs efficiently
      tensor_parallel_size: 4
      pipeline_parallel_size: 1
      expert_parallel_size: 1
      # Non-colocated: can use higher GPU memory utilization (0.7-0.8)
      # since not sharing memory with Megatron training
      gpu_memory_utilization: 0.75
      max_model_len: ${policy.max_total_sequence_length}
      enforce_eager: false
      use_deep_gemm: false
      enable_vllm_metrics_logger: true
      vllm_metrics_logger_interval: 0.5
    # compilation_config removed: use_inductor parameter not supported by installed vLLM version
    # If you upgrade vLLM to a version that supports it, you can add:
    # vllm_kwargs:
    #   compilation_config:
    #     use_inductor: false
    vllm_kwargs: {}
    colocated:
      enabled: false
      resources:
        # 1 node (4 GPUs) dedicated for vLLM generation
        gpus_per_node: 4
        num_nodes: 1

data:
  max_input_seq_length: 1024
  train:
    dataset_name: DAPOMath17K
    env_name: "math"
    processor: "math_hf_data_processor"
    prompt_file: null
    system_prompt_file: null
  validation:
    dataset_name: DAPOMathAIME2024
    env_name: "math"
    processor: "math_hf_data_processor"
    prompt_file: null
    system_prompt_file: null
  shuffle: true
  num_workers: 0

env:
  # Math env verifier implementation. "dapo_math_verify" matches the DAPO dataset ground-truth formatting.
  math:
    num_workers: 16
    math_verify_impl: "dapo_math_verify"
  # Unused by this entrypoint, but kept for compatibility with other configs.
  dapo:
    num_workers: 16
    math_verify_impl: "dapo_math_verify"

logger:
  log_dir: "logs"
  num_val_samples_to_print: 10
  wandb_enabled: false # set to true if you'd like to have WandB logging
  tensorboard_enabled: false
  mlflow_enabled: false
  swanlab_enabled: false
  monitor_gpus: false
  wandb:
    entity: nv-default-onboard
    project: "superv3-gb200-5x4-noncolocated"
    name: "dapo17k-nemotron-super-120b-gb200-5x4-noncolocated"
  gpu_monitoring:
    collection_interval: 10
    flush_interval: 10

cluster:
  # Total: 5 nodes (1 for vLLM + 4 for Megatron)
  gpus_per_node: 4
  num_nodes: 5


### B200 Config

In this part, the YAML below is tuned for a 3xB200 nodes (8xGPUs per node, 24 in total), non-colocated setup:
- **1 node** for vLLM generation (rollout)
- **2 nodes** for Megatron policy training

In [ ]:
%%writefile </YOUR/SHARED/NETWORK/STORAGE>/code/RL/examples/configs/recipes/llm/dapo17k_nemotron_super_120b_b200_3x8_noncolocated.yaml

#############################################################
# DAPO-style run on DAPO-Math-17k (pure NeMoRL) - NON-COLOCATED.
#
# - Entry point:
#     uv run examples/run_grpo_math.py --config <this.yaml>
# - Dataset:
#     HuggingFace BytedTsinghua-SIA/DAPO-Math-17k (train)
#     HuggingFace BytedTsinghua-SIA/AIME-2024 (validation)
#   via built-in NeMoRL dataset_name=DAPOMath17K
# - Cluster: 3 nodes × 8 GPUs/node (24 GPUs total)
#   - 1 node (8 GPUs) for vLLM generation
#   - 2 nodes (16 GPUs) for Megatron training
# - Model: override via CLI, e.g.
#     policy.model_name=/shared/models/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16
#     NOTE: ALL PATHS INSIDE THIS CONFIG FILE ARE PATHS AS MOUNTED INSIDE THE CONTAINER
# Notes:
# - Non-colocated setup avoids memory contention between vLLM and Megatron
# - vLLM can use higher GPU memory utilization (0.7-0.8) since not sharing with training
# - DAPO features are GRPO knobs: dynamic sampling + reward shaping (+ optional scaling).
#############################################################

defaults: ../../grpo_math_1B.yaml

grpo:
  max_num_epochs: 1
  max_num_steps: 2000

  # Rollout shape
  num_prompts_per_step: 32
  num_generations_per_prompt: 16
  max_rollout_turns: 1

  # GRPO/DAPO basics
  normalize_rewards: true
  use_leave_one_out_baseline: true
  seed: 42

  # Validation
  val_at_start: true
  val_period: 5
  # Must be non-null for `examples/run_grpo_math.py`.
  # If the validation dataset is shorter than this, it will just run through the dataset once.
  max_val_samples: 256
  val_batch_size: 32

  # ----- DAPO features -----
  use_dynamic_sampling: true
  # Dataloader batch size = batch_multiplier × num_prompts_per_step
  batch_multiplier: 2
  dynamic_sampling_max_gen_batches: 10

  # Overlong reward shaping (penalize excessively long generations)
  reward_shaping:
    enabled: true
    overlong_buffer_length: 512
    overlong_buffer_penalty: 1.0
    max_response_length: ${policy.max_total_sequence_length}

  # Optional linear reward scaling (keeps reward magnitude stable)
  reward_scaling:
    enabled: true
    source_min: 0.0
    source_max: 1.0
    target_min: -1.0
    target_max: 1.0

  # Safety rails
  overlong_filtering: false
  seq_logprob_error_threshold: 2

loss_fn:
  # DAPO clip-higher: set max > min.
  ratio_clip_min: 0.2
  ratio_clip_max: 0.28
  ratio_clip_c: null

  # Generally recommended for long-CoT RL
  token_level_loss: true

  # Keep it simple for getting started
  reference_policy_kl_penalty: 0.0
  reference_policy_kl_type: "k3"
  kl_input_clamp_value: 20.0
  kl_output_clamp_value: 10.0

  use_on_policy_kl_approximation: false
  truncated_importance_sampling_ratio: null
  use_importance_sampling_correction: false
  force_on_policy_ratio: false

checkpointing:
  enabled: true
  checkpoint_dir: "/shared/checkpoints/nemotron-superv3-dapo_math17k"
  metric_name: "val:accuracy"
  higher_is_better: true
  keep_top_k: 3
  save_period: 5
  checkpoint_must_save_by: null

policy:
  model_name: "/shared/models/NVIDIA-Nemotron-3-Super-120B-A12B-Base-BF16"
  tokenizer:
    name: ${policy.model_name}
  hf_config_overrides: {}

  # Keep batch sizes tied to rollout count (stable for GRPO)
  train_global_batch_size: ${mul:${grpo.num_prompts_per_step}, ${grpo.num_generations_per_prompt}}
  train_micro_batch_size: 1
  logprob_batch_size: 1

  # Long-context RL can be expensive; start smaller and scale up.
  max_total_sequence_length: 1024
  precision: "bfloat16"
  logprob_chunk_size: 512

  # For very large models, Megatron training is typically required.
  megatron_cfg:
    enabled: true
    error_path: "/shared/checkpoints/nemotron-superv3-dapo_math17k/logs/megatron_error.log"
    # Non-colocated: can use less aggressive memory management
    empty_unused_memory_level: 2
    activation_checkpointing: true
    tensor_model_parallel_size: 4
    pipeline_model_parallel_size: 1
    context_parallel_size: 4
    expert_tensor_parallel_size: 4
    expert_model_parallel_size: 4
    pipeline_dtype: ${policy.precision}
    sequence_parallel: true

    apply_rope_fusion: true
    defer_fp32_logits: true
    bias_activation_fusion: false

    optimizer:
      optimizer: "adam"
      lr: 1.0e-6
      min_lr: 1.0e-6
      weight_decay: 0.0
      bf16: true
      fp16: false
      params_dtype: "bfloat16"
      adam_beta1: 0.9
      adam_beta2: 0.999
      adam_eps: 1e-8
      use_distributed_optimizer: true
      use_precision_aware_optimizer: true
      clip_grad: ${policy.max_grad_norm}
      optimizer_cpu_offload: false
      optimizer_offload_fraction: 0.0

    scheduler:
      lr_decay_style: "constant"
      lr_warmup_iters: 20
      lr_warmup_init: 1.0e-7

    distributed_data_parallel_config:
      grad_reduce_in_fp32: false
      # Workaround for Megatron distributed optimizer assertion during logprob fprop:
      # `param_and_grad_buffer.py:261 assert self.param_gather_handle is None`.
      # Disabling overlap avoids starting a new param gather while a prior one is still in-flight.
      overlap_grad_reduce: false
      overlap_param_gather: false
      use_custom_fsdp: false
      data_parallel_sharding_strategy: "optim_grads_params"

  # Disable dtensor path for this getting-started config
  dtensor_cfg:
    _v2: true
    enabled: false

  sequence_packing:
    enabled: true
    train_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.train_micro_batch_size}}
    logprob_mb_tokens: ${mul:${policy.max_total_sequence_length}, ${policy.logprob_batch_size}}
    algorithm: "modified_first_fit_decreasing"
    sequence_length_round: 64

  make_sequence_length_divisible_by: ${policy.megatron_cfg.tensor_model_parallel_size}
  max_grad_norm: 1.0

  generation:
    backend: "vllm"
    max_new_tokens: ${policy.max_total_sequence_length}
    temperature: 1.0
    top_p: 1.0
    top_k: null
    stop_token_ids: null
    stop_strings: null
    vllm_cfg:
      async_engine: true
      precision: ${policy.precision}
      # vLLM on dedicated node: can use TP=4 for better performance
      # With 1 node (4 GPUs), TP=4 uses all GPUs efficiently
      tensor_parallel_size: 8
      pipeline_parallel_size: 1
      expert_parallel_size: 1
      # Non-colocated: can use higher GPU memory utilization (0.7-0.8)
      # since not sharing memory with Megatron training
      gpu_memory_utilization: 0.75
      max_model_len: ${policy.max_total_sequence_length}
      enforce_eager: false
      use_deep_gemm: false
      enable_vllm_metrics_logger: true
      vllm_metrics_logger_interval: 0.5
    # compilation_config removed: use_inductor parameter not supported by installed vLLM version
    # If you upgrade vLLM to a version that supports it, you can add:
    # vllm_kwargs:
    #   compilation_config:
    #     use_inductor: false
    vllm_kwargs: {}
    colocated:
      enabled: false
      resources:
        # 1 node (4 GPUs) dedicated for vLLM generation
        gpus_per_node: 8
        num_nodes: 1

data:
  max_input_seq_length: 1024
  train:
    dataset_name: DAPOMath17K
    env_name: "math"
    processor: "math_hf_data_processor"
    prompt_file: null
    system_prompt_file: null
  validation:
    dataset_name: DAPOMathAIME2024
    env_name: "math"
    processor: "math_hf_data_processor"
    prompt_file: null
    system_prompt_file: null
  shuffle: true
  num_workers: 0

env:
  # Math env verifier implementation. "dapo_math_verify" matches the DAPO dataset ground-truth formatting.
  math:
    num_workers: 16
    math_verify_impl: "dapo_math_verify"
  # Unused by this entrypoint, but kept for compatibility with other configs.
  dapo:
    num_workers: 16
    math_verify_impl: "dapo_math_verify"

logger:
  log_dir: "logs"
  num_val_samples_to_print: 10
  wandb_enabled: false # set to true if you'd like to have WandB logging
  tensorboard_enabled: false
  mlflow_enabled: false
  swanlab_enabled: false
  monitor_gpus: false
  wandb:
    entity: nv-default-onboard
    project: "superv3-gb200-5x4-noncolocated"
    name: "dapo17k-nemotron-super-120b-gb200-5x4-noncolocated"
  gpu_monitoring:
    collection_interval: 10
    flush_interval: 10

cluster:
  # Total: 3 nodes (1 for vLLM + 2 for Megatron)
  gpus_per_node: 8
  num_nodes: 3


## Step 2. Batch job submission 

In a production B200 cluster, each node with 8 GPUs, you can launch production batch jobs with the following procedure from a Slurm login/head node.

In [ ]:
ROOT_DIR="</YOUR/SHARED/NETWORK/STORAGE>/code/RL" # NemoRL root directory on the shared storage
RAY_SUB="${ROOT_DIR}/ray.sub" # Path to the submission file ray.sub on the host

# path to the .yaml config file inside the container
CONFIG_PATH="/shared/code/RL/examples/configs/recipes/llm/dapo17k_nemotron_super_120b_b200_3x8_noncolocated.yaml" 

export WANDB_API_KEY="YOUR_WANDB_KEY"
export ACCOUNT="YOUR_SLURM_ACCOUNT"
export PARTITION="YOUR_B200_PARTITION"
export CONTAINER="nvcr.io/<YOUR_NGC_ORG>/nemo-rl-gb200:superv3-a90de923d"
export MOUNTS="</YOUR/SHARED/NETWORK/STORAGE>:/shared"

export JOB_NAME="grpo-math-ray"
export NUM_NODES="3" # Adjust to match your compute shape
export GPUS_PER_NODE="8" # Adjust to match your compute shape
export TIME_LIMIT="02:00:00"

export HF_HOME="/shared/HF_HOME"
export NCCL_DEBUG=INFO
export NRL_VLLM_ASYNC_TIMEOUT_SECONDS=2400

export COMMAND="cd /opt/nemo-rl && uv run /opt/nemo-rl/examples/run_grpo.py --config ${CONFIG_PATH}"

sbatch \
  --nodes="${NUM_NODES}" \
  --account="${ACCOUNT}" \
  --job-name="${JOB_NAME}" \
  --partition="${PARTITION}" \
  --time="${TIME_LIMIT}" \
  --gres="gpu:${GPUS_PER_NODE}" \
  --exclusive \
  --dependency=singleton \
  "${RAY_SUB}"


On an arm-based GB200 system with multi-node NVLink (MNNVL), use the following procedure. Work with your system administrator on how to get your own [IMEX channel](https://docs.nvidia.com/multi-node-nvlink-systems/imex-guide/imexchannels.html). See further in the interactive guide below.

In [ ]:
ROOT_DIR="</YOUR/SHARED/NETWORK/STORAGE>/code/RL" # NemoRL root directory on the shared storage
RAY_SUB="${ROOT_DIR}/ray.sub" # Path to the submission file ray.sub on the host

# path to the .yaml config file inside the container
CONFIG_PATH="/shared/code/RL/examples/configs/recipes/llm/dapo17k_nemotron_super_120b_gb200_5x4_noncolocated.yaml" 

export WANDB_API_KEY="YOUR_WANDB_KEY"
export ACCOUNT="YOUR_SLURM_ACCOUNT"
export PARTITION="YOUR_B200_PARTITION"
export CONTAINER="nvcr.io/<YOUR_NGC_ORG>/nemo-rl-gb200:superv3-a90de923d"
export MOUNTS="</YOUR/SHARED/NETWORK/STORAGE>:/shared"

export JOB_NAME="grpo-math-ray"
export NUM_NODES="5" # Adjust to match your compute shape
export GPUS_PER_NODE="4" # Adjust to match your compute shape
export TIME_LIMIT="02:00:00"

export HF_HOME="/shared/HF_HOME"
export NCCL_DEBUG=INFO
export NRL_VLLM_ASYNC_TIMEOUT_SECONDS=2400
export NCCL_MNNVL_ENABLE=1
export GLOO_SOCKET_IFNAME="enP5p9s0" # Supply your primary eth interface here
export NVIDIA_IMEX_CHANNELS=2046 # Supply your correct IMEX channel here

export COMMAND="cd /opt/nemo-rl && uv run /opt/nemo-rl/examples/run_grpo.py --config ${CONFIG_PATH}"

sbatch \
  --nodes="${NUM_NODES}" \
  --account="${ACCOUNT}" \
  --job-name="${JOB_NAME}" \
  --partition="${PARTITION}" \
  --time="${TIME_LIMIT}" \
  --gres="gpu:${GPUS_PER_NODE}" \
  --exclusive \
  --dependency=singleton \
  "${RAY_SUB}"

**Tweaking training hyperparameters**

Once training verification is successful, you can tweak the configuration parameters:

- **Run length / throughput (`grpo`)**
  - `max_num_steps`, `num_prompts_per_step`, `num_generations_per_prompt`
  - These directly control training duration, sample volume, and compute cost per step.

- **Reward behavior (`grpo.reward_shaping`, `grpo.reward_scaling`)**
  - `reward_shaping.enabled`, `overlong_buffer_length`, `overlong_buffer_penalty`
  - `reward_scaling.enabled` and min/max ranges
  - Use these to penalize overly long outputs and keep reward magnitude stable.

- **Policy sequence budget (`policy`)**
  - `max_total_sequence_length`, `train_micro_batch_size`, `logprob_batch_size`
  - Increase carefully: longer contexts improve reasoning capacity but significantly raise memory and latency.

- **Distributed training topology (`policy.megatron_cfg`)**
  - `tensor_model_parallel_size`, `pipeline_model_parallel_size`, `context_parallel_size`
  - Must match your available training GPUs and desired DP/TP/PP/CP balance.

- **Generation backend (`policy.generation.vllm_cfg`)**
  - `tensor_parallel_size`, `gpu_memory_utilization`, `max_model_len`
  - Tune for rollout speed and stability on the dedicated generation node.

- **Dataset and validation (`data`, `grpo`)**
  - `data.dataset_name`, `data.validation.dataset_name`, `max_val_samples`, `val_period`
  - Keep validation frequent enough to catch regressions, but not so frequent that it slows training.


# Interactive Ray training cluster setup (Optional)

In a production setting, it is more convenient to submit training jobs as Slurm batch jobs. However, setting up an interactive training environment allows you to iterate and debug faster. Once the interactive jobs run smoothly, you can submit them as batch training jobs.

The rest of this guide outlines this optional path where you can set up an interactive persistent Ray cluster for quick development.
 
## Step 4. Reserve worker nodes and start Docker containers

Reserve **5 interactive B200/GB200 nodes** as worker nodes for your Ray cluster. If using GB200 nodes on a GB200NVL72 cluster, ideally these nodes should reside within the same  rack. This rack-colocation of nodes will enable fast interconnect for faster training, in particular, [multi-node NVLink](https://docs.nvidia.com/multi-node-nvlink-systems/mnnvl-user-guide/overview.html).

**Note**: 
- Work with your system administrator on how to get your own [IMEX channel](https://docs.nvidia.com/multi-node-nvlink-systems/imex-guide/imexchannels.html). IMEX stands for Internode Memory Exchange/Management. It is a system service used in multi-node NVLink environments (like DGX or HGX systems) to securely facilitate sharing GPU memory between different nodes (servers) over the NVLink fabric. IMEX channels are a GPU driver feature that allows for user-based memory isolation in a multi-user environment within an IMEX domain.

- On a GB200 node you can check your IMEX channel with
```
root@gb200-compute01:/home# ls /dev/nvidia-caps-imex-channels/
channel1035
```
In this example, your IMEX channel is hence 1035.

- For GB200NVL72 systems, work with your system administrator on how to identify and reserve nodes on the same rack. One possible way is via node naming in Slurm. For example, nodes in the same rack could be named in consecutive numbers sharing the same prefix, such as `gb200-001-compute[01-09]`. In this case, reserve nodes while specifying an explicit node name, such as:
    ```
    srun -p gb200nvl72 -w gb200-001-compute01 -t 08:00:00 --pty bash
    ```


From each of the 5 interactive bash sessions on 5 reserved GB200 nodes, start the Docker container while mounting the shared directory as follows:

In [ ]:
# Run on each of the 5 nodes, supply your own NVIDIA_IMEX_CHANNELS id and YOUR_NGC_ORG here
docker run -it --rm \
  --runtime=nvidia \
  -e NVIDIA_VISIBLE_DEVICES=0,1,2,3 \
  -e NVIDIA_IMEX_CHANNELS=1035 \ # Supply your own IMEX channel ID
  --gpus=all \
  --ipc=host \
  --net=host \
  --shm-size=64gb \
  -v </YOUR/SHARED/NETWORK/STORAGE>:/shared \
  nvcr.io/<YOUR_NGC_ORG>/nemo-rl-gb200:superv3-13559c73 \
  bash

## Step 5. Manually start the Ray cluster

First, check the IPs of your nodes.

In [ ]:
# Check which nodes are allocated to you
squeue | grep <your_user_name>

# Verify connectivity and get IP addresses
ping <node_name>

Dedicate **1 node** as the head node and start the Ray master process. From within the NemoRL docker container of the head node, replacing the IP with the correct IP of your head node:

In [ ]:
# Master node
ray start --head --node-ip-address=100.121.33.1 --port=6379

Then, manually start the Ray cluster on each worker node and connect to the head node, replacing the IPs with the correct IP of your head node and worker nodes:

In [ ]:
# Worker 1
ray start --address='100.121.33.1:6379' --node-ip-address=100.121.33.2

# Worker 2
ray start --address='100.121.33.1:6379' --node-ip-address=100.121.33.3

# Worker 3
ray start --address='100.121.33.1:6379' --node-ip-address=100.121.33.4

# Worker 4
ray start --address='100.121.33.1:6379' --node-ip-address=100.121.33.5

From the Ray head node, validate your cluster setup with


In [ ]:
ray status

If successful, this should list 5 available nodes with 20 GPUs in total. This persistent Ray cluster allows you to do quick-turnaround interactive development, shortening the dev-debug cycle.

```
======== Autoscaler status: 2026-03-03 03:47:10.276939 ========
Node status
---------------------------------------------------------------
Active:
 1 node_a906c992db87ec9c1990ad57e63a191cf183e0f578e2e70f8d348fb1
 1 node_a4b3453d98dccb507f5809a9f2d908c538869de51e7b29a830d7fc02
 1 node_18d83de59b62de00e8f66b6a2b40a1a94e95abdc14fc1ad3120f4763
 1 node_de79694e19e94d3ca67ef19392868e297d461b96a3bd276ec7d01334
 1 node_7c2c04cc7bb9b79320a8dcb8932016936840849d927998b97c303075
Pending:
 (no pending nodes)
Recent failures:
 (no failures)

Resources
---------------------------------------------------------------
Total Usage:
 0.0/720.0 CPU
 0.0/20.0 GPU
 0B/3.79TiB memory
 0B/931.32GiB object_store_memory

Total Constraints:
 (no request_resources() constraints)
Total Demands:
 (no resource demands)
 ```

## Step 6. Start Interactive RL Training

Now launch the RL workload from the Ray head node using the config prepared in Step 4.

**Notes**:
- Set `HF_HOME` to a high-speed shared location so datasets and converted checkpoints are not stored in a small home directory.
- Set `WANDB_API_KEY` if you want experiment tracking on [Wandb](https://wandb.ai/).
- Set `GLOO_SOCKET_IFNAME` to your primary ethernet interface (check with `ifconfig`) to avoid distributed-communication issues.

From the Ray head node, submit the interactive job with:

In [ ]:
export WANDB_API_KEY="YOUR_WANDB_KEY"
export HF_HOME="/shared/HF_HOME" # Your HF cache directory on the shared storage

export NCCL_DEBUG=INFO
export NCCL_MNNVL_ENABLE=1 # Enable multi-node NVLink on GB200 system
export GLOO_SOCKET_IFNAME="enP5p9s0"
export NRL_VLLM_ASYNC_TIMEOUT_SECONDS=2400

uv run /opt/nemo-rl/examples/run_grpo.py \
  --config /shared/code/RL/examples/configs/recipes/llm/dapo17k_nemotron_super_120b_gb200_5x4_noncolocated.yaml

This will launch an interactive training job with 5 given nodes, 1 for vLLM roll out and 4 for policy training using the Megatron backend.

